# Aegis: build training graphs and train on Kaggle

End-to-end run: BigVul functions → leak-free split → Joern code property graphs → CodeBERT node features → HGT model → PR-AUC / F1.

**Before running (notebook sidebar → Settings):**
1. **Accelerator:** GPU (T4 or P100).
2. **Internet:** on (needed for pip, Java, Joern and the CodeBERT download).
3. **Input:** add a Kaggle dataset containing `MSR_data_cleaned.csv` (BigVul). The notebook finds it anywhere under `/kaggle/input`.
4. **If the GitHub repo is private:** add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) holding a read-only token for the repo.

**Resuming after a session ends:** both steps resume where they stopped, but only if their output directories survive. `/kaggle/working` is kept when you *Save Version*; to continue in a new session, add the previous version's output as an input and run the *Restore previous output* cell.

In [ ]:
SAMPLE_SIZE = 2000      # functions to use; set to None for the full dataset (~188K)
BATCH_SIZE = 500        # functions parsed per Joern invocation
EPOCHS = 30
SEED = 42

REPO_URL = "github.com/Hikunokyojin/aegis.git"
JOERN_VERSION = "v4.0.597"
WORK = "/kaggle/working"
GRAPHS_DIR = f"{WORK}/graphs_{SAMPLE_SIZE or 'full'}"
RUN_DIR = f"{WORK}/runs/sample_{SAMPLE_SIZE or 'full'}"

In [ ]:
# Get the code: uses the GITHUB_TOKEN secret if present (private repo), otherwise a plain clone.
import os, subprocess

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass

url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"
if not os.path.exists(f"{WORK}/aegis"):
    subprocess.run(["git", "clone", "--depth", "1", url, f"{WORK}/aegis"], check=True)
print(subprocess.run(["git", "-C", f"{WORK}/aegis", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
# Install the project with the exact dependency versions it was developed and tested with.
!pip install -q -e /kaggle/working/aegis

import torch, torch_geometric, transformers
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("torch_geometric", torch_geometric.__version__, "| transformers", transformers.__version__)
# If CUDA shows False here with the GPU accelerator on, the pinned torch wheel doesn't match
# Kaggle's GPU driver: reinstall with `pip install -q --no-deps -e /kaggle/working/aegis`
# followed by `pip install -q torch-geometric tree-sitter tree-sitter-cpp==0.23.4 tree-sitter-java==0.23.5`
# to keep Kaggle's preinstalled torch instead.

In [ ]:
# Java 21 + Joern (same versions as the local setup the pipeline was verified against).
!apt-get -qq update && apt-get -qq install -y openjdk-21-jre-headless > /dev/null
!wget -q https://github.com/joernio/joern/releases/download/{JOERN_VERSION}/joern-cli-linux-x86_64.zip -O /tmp/joern.zip
!unzip -q -o /tmp/joern.zip -d /tmp/joern && rm /tmp/joern.zip

import glob
joern_dir = os.path.dirname(glob.glob("/tmp/joern/**/joern-parse", recursive=True)[0])
os.environ["PATH"] = joern_dir + ":" + os.environ["PATH"]
!java -version 2>&1 | head -1
!joern-parse --help | head -2

In [ ]:
# Restore previous output (only when continuing an earlier run): copies an earlier version's
# graphs/ and runs/ from /kaggle/input so both steps below resume instead of starting over.
import shutil
for previous in glob.glob("/kaggle/input/*/graphs_*") + glob.glob("/kaggle/input/*/runs"):
    target = f"{WORK}/{os.path.basename(previous)}"
    if not os.path.exists(target):
        shutil.copytree(previous, target)
        print("restored", previous, "->", target)

In [ ]:
# Build training graphs. Resumable: re-running skips functions whose graph is already saved.
csv_matches = glob.glob("/kaggle/input/**/MSR_data_cleaned.csv", recursive=True)
assert csv_matches, "Add the BigVul dataset (MSR_data_cleaned.csv) as a notebook input first."
BIGVUL_CSV = csv_matches[0]

sample_arg = f"--sample-size {SAMPLE_SIZE}" if SAMPLE_SIZE else ""
!aegis-build-training-graphs --bigvul-csv "{BIGVUL_CSV}" --output-dir "{GRAPHS_DIR}" \
    --batch-size {BATCH_SIZE} --seed {SEED} {sample_arg} 2>&1 | grep -viE "warn|edges skipped"

In [ ]:
# How many vulnerable functions each split actually got (a split with none can't be scored).
import json
print(open(f"{GRAPHS_DIR}/split.json").read())
for split in ("train", "val", "test"):
    graphs = glob.glob(f"{GRAPHS_DIR}/{split}/graphs/*.pt")
    labels = [torch.load(g, weights_only=False)["label"] for g in graphs]
    print(f"{split:5s} {len(labels):6d} graphs, {sum(labels):5d} vulnerable")

In [ ]:
# Train. Resumable: re-running continues from the last finished epoch.
!aegis-train --data-dir "{GRAPHS_DIR}" --run-dir "{RUN_DIR}" --epochs {EPOCHS} 2>&1 | grep -viE "warn"

In [ ]:
# Score the best checkpoint (chosen on validation PR-AUC) on the held-out test split.
from pathlib import Path
from aegis.model.hetero_gnn import AegisHeteroGNN
from aegis.training.trainer import evaluate, load_split

device = "cuda" if torch.cuda.is_available() else "cpu"
best = torch.load(f"{RUN_DIR}/best.pt", weights_only=False, map_location=device)
model = AegisHeteroGNN(**best["config"]).to(device)
model.load_state_dict(best["model"])

test_metrics = evaluate(model, load_split(Path(GRAPHS_DIR) / "test" / "graphs"))
print(f"best epoch {best['epoch']}, validation PR-AUC {best['val_pr_auc']:.4f}")
print("test:", test_metrics)
json.dump({"best_epoch": best["epoch"], "val_pr_auc": best["val_pr_auc"], "test": test_metrics},
          open(f"{RUN_DIR}/test_results.json", "w"), indent=2)